In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

In [ ]:
system_message = """ 
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, just say so. """

In [ ]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
#Tools - get ticket price for a certain location

# = {"london":"$800", "paris":"$900", "tokyo":"$1000", "berlin":"$500"}

def get_ticket_price(destination_city):
    print(f"tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(),"unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}."

In [ ]:
get_ticket_price("Paris")

In [ ]:
#JSON structure required to describe this function .

price_function = {
    "name" : "get_ticket_price",
    "description" : "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties":{
            "destination_city":{
                "type":"string",
                "description" : "The city that the customer wants to travel to",
            },
        },
        "required":["destination_city"],
        "additionalProperties":False
    }
}

In [ ]:
# a list of tools
tools = [{"type" : "function", "function": price_function}, {"type":"function", "function": set_price_function}]

In [ ]:
tools

In [ ]:



def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message=response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)

#Check how the tool calls are returned 
    #for message in messages:
        #print(message)

    return response.choices[0].message.content    
    

In [ ]:
# function handle_tool_call

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name =="get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role":"tool",
            "content":price_details,
            "tool_call_id":tool_call.id
        }
    return response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
# Refine handle tool calls to iterate for tools - instead of just one choice.

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
# while not looping through, if will just send one message response. A multi statement will require a loop

    if response.choices[0].finish_reason=="tool_calls":
        message=response.choices[0].message
        response = handle_tool_calls(message)
        messages.append(message)
        messages.extend(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)

#Check how the tool calls are returned 
    #for message in messages:
        #print(message)

    return response.choices[0].message.content    

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id })

        elif tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            price = arguments.get('destination_city')
            set_price_details = set_ticket_price(city,price)
            responses.append({
                "role": "tool",
                "content": set_price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
# changing while instead of if so the messages can loop through , LLM will handle this so no need to worry of infinite loop

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)


    while response.choices[0].finish_reason=="tool_calls":
        message=response.choices[0].message
        response = handle_tool_calls(message)
        messages.append(message)
        messages.extend(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages,tools=tools)

#Check how the tool calls are returned 
    #for message in messages:
        #print(message)

    return response.choices[0].message.content 

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
import sqlite3

In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
# new get_ticket_price with sqlite3 connection
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM prices WHERE city = ?",(city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is {result[0]}" if result else "No price data available for this city"

In [ ]:
get_ticket_price("London")

In [ ]:
#function to insert prices to our database

def set_ticket_price(city,price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city,price) VALUES (?,?)ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
        


In [ ]:
ticket_prices = {"london":799, "paris" : 899, "tokyo": 1200, "berlin": 900, "sydney": 1200}
for city,price in ticket_prices.items():
    set_ticket_price(city,price)

In [ ]:
get_ticket_price("sydney")

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
set_price_function = {
    "name" : "set_ticket_price",
    "description" : "Set the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties":{
            "destination_city":{
                "type":"string",
                "description" : "The city that the customer wants to travel to",
            },
        },
        "required":["destination_city"],
        "additionalProperties":False
    }
}